# Step 11: Regression Multi-Horizon - Lot-Normalized Deviations
## Predict Vt deviation from lot mean (z-score) instead of raw values

This notebook trains CatBoostRegressor models to predict **lot-normalized Vt deviations** instead of raw Vt values.

**Key difference from notebook 10:**
- **Target**: `(Vt - lot_mean) / lot_std` (z-score within lot)
- **Rationale**: Features capture *relative* process behavior, not absolute baselines
- **Hypothesis**: Lot-level baseline variation is unpredictable from sensor features, but deviations from baseline are

This aligns with how the classification model works (outlier detection uses lot-level IQR).

## 1. Setup and Configuration

In [1]:
import time
import json
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

start_time = time.time()
print("=" * 60)
print("Step 11: Regression Multi-Horizon - Lot-Normalized")
print("=" * 60)

Step 11: Regression Multi-Horizon - Lot-Normalized


In [2]:
# AUTO-DETECT PROJECT ROOT
import os
from pathlib import Path

def find_project_root(marker_file='CLAUDE.md', max_depth=5):
    """Find project root by looking for marker file"""
    current = Path.cwd()
    
    # Try current directory and parents
    for _ in range(max_depth):
        if (current / marker_file).exists():
            return current
        current = current.parent
    
    # If not found, return current working directory
    print(f"WARNING: Could not find {marker_file}, using current directory")
    return Path.cwd()

# Find and change to project root
project_root = find_project_root()
os.chdir(project_root)

print(f"Project root: {project_root}")
print(f"Current directory: {Path.cwd()}")
print(f"\nData directory exists: {(Path.cwd() / 'data').exists()}")
print(f"Outputs directory exists: {(Path.cwd() / 'outputs').exists()}")

Project root: d:\capstone_pipeline
Current directory: d:\capstone_pipeline

Data directory exists: True
Outputs directory exists: True


In [3]:
# CONFIGURATION
vt_type = 'NFET1_VT'  # Options: NFET1_VT, PFET2_VT, NFET2_VT, PFET1_VT
force = False  # Set to True to force rebuild

print(f"\n[CONFIG] Vt type: {vt_type}")
print(f"[CONFIG] Force rebuild: {force}")


[CONFIG] Vt type: NFET1_VT
[CONFIG] Force rebuild: False


In [4]:
# Setup paths
data_dir = Path("data")
output_dir = Path("outputs")
features_dir = output_dir / "features"
models_dir = output_dir / "models"
plots_dir = output_dir / "plots"
models_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

sentinel = models_dir / "11_regression_lot_normalized.done"
if sentinel.exists() and not force:
    print(f"\n[OK] Lot-normalized regression models already trained (found {sentinel})")
    print(f"  Set force=True to rebuild")
else:
    print(f"\n[INFO] Training lot-normalized regression models...")


[INFO] Training lot-normalized regression models...


## 2. Load and Prepare Lot-Normalized Regression Target

In [5]:
# Check required files
required_files = [
    data_dir / "response_updated.csv",
    features_dir / "train.parquet",
    features_dir / "val.parquet",
    features_dir / "column_step_map.json"
]

for file in required_files:
    if not file.exists():
        raise FileNotFoundError(
            f"Required file not found: {file}\n"
            f"Please run previous pipeline steps first."
        )

print("[OK] All required files found")

[OK] All required files found


In [6]:
# Load regression targets and compute LOT-NORMALIZED deviations
print("\nLoading and normalizing regression targets...")
response_df = pd.read_csv(data_dir / "response_updated.csv")
print(f"  Total rows in response_updated.csv: {len(response_df)}")

# Filter to selected Vt type
vt_filtered = response_df[response_df['NAME'] == vt_type][['WAFER_SCRIBE', 'LOT_ID', 'VALUE']].copy()
print(f"  Rows with {vt_type}: {len(vt_filtered)}")
print(f"  Unique wafers: {vt_filtered['WAFER_SCRIBE'].nunique()}")

# Handle duplicate measurements by averaging per wafer
duplicates = len(vt_filtered) - vt_filtered['WAFER_SCRIBE'].nunique()
if duplicates > 0:
    print(f"  Found {duplicates} duplicate measurements - averaging per wafer")
    # Average duplicates while preserving LOT_ID
    vt_filtered = vt_filtered.groupby(['WAFER_SCRIBE', 'LOT_ID'], as_index=False)['VALUE'].mean()

# Compute lot-level statistics
print("\n  Computing lot-level statistics...")
lot_stats = vt_filtered.groupby('LOT_ID')['VALUE'].agg(['mean', 'std']).reset_index()
lot_stats.columns = ['LOT_ID', 'lot_mean', 'lot_std']

# Handle lots with zero std (all wafers have identical Vt)
zero_std_lots = (lot_stats['lot_std'] == 0) | (lot_stats['lot_std'].isna())
if zero_std_lots.any():
    print(f"  WARNING: {zero_std_lots.sum()} lots have zero std - setting std=1.0 to avoid division by zero")
    lot_stats.loc[zero_std_lots, 'lot_std'] = 1.0

# Merge lot stats and compute normalized deviation (z-score)
vt_df = vt_filtered.merge(lot_stats, on='LOT_ID', how='left')
vt_df['vt_deviation'] = (vt_df['VALUE'] - vt_df['lot_mean']) / vt_df['lot_std']

# Keep only necessary columns
vt_df = vt_df[['WAFER_SCRIBE', 'LOT_ID', 'vt_deviation']].copy()

print(f"\n  Final unique wafers: {len(vt_df)}")
print(f"  Deviation range: [{vt_df['vt_deviation'].min():.4f}, {vt_df['vt_deviation'].max():.4f}]")
print(f"  Deviation mean: {vt_df['vt_deviation'].mean():.4f} (should be ~0)")
print(f"  Deviation std: {vt_df['vt_deviation'].std():.4f} (should be ~1)")

# Verify normalization worked
if abs(vt_df['vt_deviation'].mean()) > 0.01 or abs(vt_df['vt_deviation'].std() - 1.0) > 0.05:
    print("\n  WARNING: Deviation statistics not well-normalized!")
    print("  This may indicate issues with lot-level grouping")


Loading and normalizing regression targets...
  Total rows in response_updated.csv: 76644
  Rows with NFET1_VT: 17202
  Unique wafers: 16817
  Found 385 duplicate measurements - averaging per wafer

  Computing lot-level statistics...

  Final unique wafers: 16821
  Deviation range: [-3.4949, 3.1121]
  Deviation mean: 0.0000 (should be ~0)
  Deviation std: 0.9758 (should be ~1)


In [7]:
# Load existing feature matrices
print("\nLoading feature matrices...")
train_df = pd.read_parquet(features_dir / "train.parquet")
val_df = pd.read_parquet(features_dir / "val.parquet")
print(f"  Train size: {len(train_df)}")
print(f"  Val size: {len(val_df)}")


Loading feature matrices...
  Train size: 13510
  Val size: 3307


In [8]:
# Merge lot-normalized target with feature matrices
print("\nMerging lot-normalized targets with features...")
train_size_before = len(train_df)
val_size_before = len(val_df)

train_df = train_df.merge(vt_df, on='WAFER_SCRIBE', how='left')
val_df = val_df.merge(vt_df, on='WAFER_SCRIBE', how='left')

# Verify merge didn't create duplicates
train_size_after = len(train_df)
val_size_after = len(val_df)
if train_size_after != train_size_before or val_size_after != val_size_before:
    raise ValueError(
        f"Merge created duplicates! Train: {train_size_before} -> {train_size_after}, "
        f"Val: {val_size_before} -> {val_size_after}. "
        f"This indicates duplicate WAFER_SCRIBE values in vt_df."
    )
print(f"  Merge successful (no duplicates created)")

# Drop original classification target if present
if 'is_outlier' in train_df.columns:
    train_df = train_df.drop(columns=['is_outlier'])
    val_df = val_df.drop(columns=['is_outlier'])

# Handle LOT_ID column duplication from merge
if 'LOT_ID_x' in train_df.columns and 'LOT_ID_y' in train_df.columns:
    # Keep LOT_ID_x (from feature matrix) and drop LOT_ID_y (from vt_df)
    train_df = train_df.drop(columns=['LOT_ID_y']).rename(columns={'LOT_ID_x': 'LOT_ID'})
    val_df = val_df.drop(columns=['LOT_ID_y']).rename(columns={'LOT_ID_x': 'LOT_ID'})

# Check for missing values in target
train_missing = train_df['vt_deviation'].isna().sum()
val_missing = val_df['vt_deviation'].isna().sum()
print(f"  Missing targets in train: {train_missing} ({100*train_missing/len(train_df):.2f}%)")
print(f"  Missing targets in val: {val_missing} ({100*val_missing/len(val_df):.2f}%)")

# Drop rows with missing targets
if train_missing > 0 or val_missing > 0:
    train_df = train_df.dropna(subset=['vt_deviation'])
    val_df = val_df.dropna(subset=['vt_deviation'])
    print(f"  Dropped {train_missing} train and {val_missing} val rows with missing targets")

print(f"  Final train size: {len(train_df)}")
print(f"  Final val size: {len(val_df)}")


Merging lot-normalized targets with features...


ValueError: Merge created duplicates! Train: 13510 -> 13514, Val: 3307 -> 3307. This indicates duplicate WAFER_SCRIBE values in vt_df.

## 3. Load Column-Step Mapping

In [9]:
# Load column-to-step mapping
print("\nLoading column-step mapping...")
with open(features_dir / "column_step_map.json", 'r') as f:
    column_step_map = json.load(f)

print(f"  Loaded mappings for {len(column_step_map)} features")
print(f"  SeqNo range: {min(column_step_map.values())} to {max(column_step_map.values())}")


Loading column-step mapping...
  Loaded mappings for 10302 features
  SeqNo range: 0 to 240


In [10]:
# Load step sequence to define horizons
print("\nDefining horizons...")
step_seq_file = data_dir / "step_seq.csv"

if not step_seq_file.exists():
    print(f"  WARNING: {step_seq_file} not found.")
    print(f"  Using SeqNo values from column_step_map instead.")
    all_seqnos = sorted(set(column_step_map.values()))
    all_seqnos = [s for s in all_seqnos if s > 0]  # Exclude SeqNo=0
else:
    step_seq_df = pd.read_csv(step_seq_file)
    all_seqnos = sorted(step_seq_df['SeqNo'].unique())
    print(f"  Loaded {len(all_seqnos)} step sequence numbers")


Defining horizons...
  Loaded 240 step sequence numbers


In [11]:
# Define horizons at deciles (10%, 20%, ..., 100% of process)
percentiles = np.percentile(all_seqnos, [10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
horizons = sorted(set([int(p) for p in percentiles]))

print(f"\n  Training models at {len(horizons)} horizons:")
for h in horizons:
    pct = 100 * h / max(all_seqnos)
    print(f"    Horizon {h:>4} ({pct:>5.1f}% complete)")


  Training models at 10 horizons:
    Horizon   24 ( 10.0% complete)
    Horizon   48 ( 20.0% complete)
    Horizon   72 ( 30.0% complete)
    Horizon   96 ( 40.0% complete)
    Horizon  120 ( 50.0% complete)
    Horizon  144 ( 60.0% complete)
    Horizon  168 ( 70.0% complete)
    Horizon  192 ( 80.0% complete)
    Horizon  216 ( 90.0% complete)
    Horizon  240 (100.0% complete)


In [12]:
# Prepare metadata columns
metadata_cols = ['WAFER_SCRIBE', 'LOT_ID', 'vt_deviation', 'PARAM_END_DATETIME', 
                'first_start_time']
feature_cols = [col for col in train_df.columns if col not in metadata_cols]

print(f"\n  Total available features: {len(feature_cols)}")


  Total available features: 10305


In [13]:
# Extract regression targets (lot-normalized deviations)
y_train = train_df['vt_deviation']
y_val = val_df['vt_deviation']

print(f"\n  Target statistics (train):")
print(f"    Mean: {y_train.mean():.4f} (should be ~0)")
print(f"    Std: {y_train.std():.4f} (should be ~1)")
print(f"    Range: [{y_train.min():.4f}, {y_train.max():.4f}]")
print(f"\n  Target statistics (val):")
print(f"    Mean: {y_val.mean():.4f} (should be ~0)")
print(f"    Std: {y_val.std():.4f} (should be ~1)")
print(f"    Range: [{y_val.min():.4f}, {y_val.max():.4f}]")


  Target statistics (train):
    Mean: 0.0000 (should be ~0)
    Std: 0.9759 (should be ~1)
    Range: [-3.4949, 3.1121]

  Target statistics (val):
    Mean: -0.0000 (should be ~0)
    Std: 0.9755 (should be ~1)
    Range: [-2.9101, 2.8513]


## 4. Multi-Horizon Regression Training Loop

In [ ]:
# Train regression models at each horizon
horizon_results = []

print("\n" + "=" * 80)
print("Training lot-normalized regression horizon models...")
print("=" * 80)
print(f"{'Horizon':<10} {'Features':<10} {'RMSE':<12} {'MAE':<12} {'R²':<10}")
print("-" * 80)

for k in tqdm(horizons, desc="Horizons"):
    # Filter features to those available at horizon k
    available_features = [
        col for col in feature_cols 
        if col in column_step_map and column_step_map[col] <= k
    ]
    
    if len(available_features) == 0:
        print(f"\n  WARNING: No features available at horizon {k}, skipping...")
        continue
    
    # Prepare feature matrices
    X_train = train_df[available_features].copy()
    X_val = val_df[available_features].copy()
    
    # Identify categorical features in this subset
    cat_features = []
    for col in available_features:
        is_categorical = (
            X_train[col].dtype == 'object' or
            X_train[col].dtype.name == 'string' or
            str(X_train[col].dtype).startswith('str') or
            col.startswith('LOT_ID') or
            '__EQUIP' in col or
            '__POSITION' in col or
            col in ['first_step', 'last_step']
        )
        if is_categorical:
            cat_features.append(col)
            # Ensure categorical columns are strings
            X_train[col] = X_train[col].fillna('UNKNOWN').astype(str)
            X_val[col] = X_val[col].fillna('UNKNOWN').astype(str)
    
    # Create CatBoost pools
    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    val_pool = Pool(X_val, y_val, cat_features=cat_features)
    
    # Train regression model
    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function='RMSE',
        eval_metric='RMSE',
        early_stopping_rounds=30,
        random_seed=42,
        verbose=0  # Suppress per-iteration output
    )
    
    model.fit(train_pool, eval_set=val_pool)
    
    # Evaluate
    y_pred = model.predict(X_val)
    
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    # Save model
    model_file = models_dir / f"regression_lot_normalized_horizon_{k:03d}_model.cbm"
    model.save_model(str(model_file))
    
    # Store results
    horizon_results.append({
        'horizon': int(k),
        'n_features': len(available_features),
        'rmse': float(rmse),
        'mae': float(mae),
        'r2': float(r2)
    })
    
    # Print summary line
    print(f"k={k:<7} {len(available_features):<10} {rmse:<12.6f} {mae:<12.6f} {r2:<10.4f}")

print("-" * 80)


Training lot-normalized regression horizon models...
Horizon    Features   RMSE         MAE          R²        
--------------------------------------------------------------------------------


Horizons:   0%|          | 0/10 [00:00<?, ?it/s]

## 5. Save Results and Metrics

In [ ]:
# Save results to JSON
results_file = output_dir / "regression_lot_normalized_horizon_results.json"
with open(results_file, 'w') as f:
    json.dump({
        'vt_type': vt_type,
        'target': 'lot_normalized_deviation',
        'horizons': horizon_results
    }, f, indent=2)

print(f"\n[OK] Saved {results_file.name} with {len(horizon_results)} entries")

In [ ]:
# Summary statistics
print("\n" + "=" * 60)
print("Lot-Normalized Regression Model Training Summary")
print("=" * 60)
print(f"Vt type: {vt_type}")
print(f"Target: Lot-normalized deviation (z-score)")
print(f"Total horizons evaluated: {len(horizon_results)}")
print(f"Horizon range: {min(r['horizon'] for r in horizon_results)} to "
      f"{max(r['horizon'] for r in horizon_results)}")
print(f"\nPerformance range:")
print(f"  RMSE: {min(r['rmse'] for r in horizon_results):.6f} to "
      f"{max(r['rmse'] for r in horizon_results):.6f}")
print(f"  MAE:  {min(r['mae'] for r in horizon_results):.6f} to "
      f"{max(r['mae'] for r in horizon_results):.6f}")
print(f"  R²:   {min(r['r2'] for r in horizon_results):.4f} to "
      f"{max(r['r2'] for r in horizon_results):.4f}")

# Find first viable prediction point (R² > 0.6)
viable_horizons = [r for r in horizon_results if r['r2'] > 0.6]
if viable_horizons:
    first_viable = min(viable_horizons, key=lambda r: r['horizon'])
    max_seqno = max(r['horizon'] for r in horizon_results)
    pct_complete = 100 * first_viable['horizon'] / max_seqno
    print(f"\nFirst viable prediction point (R² > 0.6):")
    print(f"  Horizon {first_viable['horizon']} ({pct_complete:.1f}% complete)")
    print(f"  R²: {first_viable['r2']:.4f}")
    print(f"  RMSE: {first_viable['rmse']:.6f}")
    print(f"  MAE: {first_viable['mae']:.6f}")
    print(f"  Features: {first_viable['n_features']}")
else:
    print(f"\nNo horizon achieved R² > 0.6")
    best_horizon = max(horizon_results, key=lambda r: r['r2'])
    print(f"Best R² achieved: {best_horizon['r2']:.4f} at horizon {best_horizon['horizon']}")

## 6. Evaluation Plots (Final Horizon)

In [ ]:
# Load best/final model for detailed evaluation
final_horizon = max(r['horizon'] for r in horizon_results)
final_model_file = models_dir / f"regression_lot_normalized_horizon_{final_horizon:03d}_model.cbm"

print(f"\n[INFO] Loading final model from horizon {final_horizon} for evaluation...")
from catboost import CatBoostRegressor
final_model = CatBoostRegressor()
final_model.load_model(str(final_model_file))

# Get available features for final horizon
final_available_features = [
    col for col in feature_cols 
    if col in column_step_map and column_step_map[col] <= final_horizon
]

X_val_final = val_df[final_available_features].copy()

# Handle categorical features
cat_features_final = []
for col in final_available_features:
    is_categorical = (
        X_val_final[col].dtype == 'object' or
        X_val_final[col].dtype.name == 'string' or
        str(X_val_final[col].dtype).startswith('str') or
        col.startswith('LOT_ID') or
        '__EQUIP' in col or
        '__POSITION' in col or
        col in ['first_step', 'last_step']
    )
    if is_categorical:
        cat_features_final.append(col)
        X_val_final[col] = X_val_final[col].fillna('UNKNOWN').astype(str)

# Get predictions
y_pred_final = final_model.predict(X_val_final)

print(f"[OK] Predictions generated for {len(y_pred_final)} validation samples")

In [ ]:
# Plot 1: Predicted vs Actual (Deviations)
print("\n[INFO] Generating Predicted vs Actual plot (lot-normalized deviations)...")
plt.figure(figsize=(10, 8))
plt.scatter(y_val, y_pred_final, alpha=0.5, s=20)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Vt Deviation (z-score)', fontsize=12)
plt.ylabel('Predicted Vt Deviation (z-score)', fontsize=12)
plt.title(f'Predicted vs Actual Vt Deviation ({vt_type}) - Horizon {final_horizon}\nR² = {r2_score(y_val, y_pred_final):.4f}', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
pred_vs_actual_file = plots_dir / "regression_lot_normalized_pred_vs_actual.png"
plt.savefig(pred_vs_actual_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"[OK] Saved to {pred_vs_actual_file}")

In [ ]:
# Plot 2: Residual Plot
print("\n[INFO] Generating Residual plot...")
residuals = y_val.values - y_pred_final
plt.figure(figsize=(10, 6))
plt.scatter(y_pred_final, residuals, alpha=0.5, s=20)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Vt Deviation (z-score)', fontsize=12)
plt.ylabel('Residuals (Actual - Predicted)', fontsize=12)
plt.title(f'Residual Plot - Lot-Normalized ({vt_type}) - Horizon {final_horizon}', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
residual_file = plots_dir / "regression_lot_normalized_residuals.png"
plt.savefig(residual_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"[OK] Saved to {residual_file}")

In [ ]:
# Plot 3: Feature Importance
print("\n[INFO] Generating Feature Importance plot...")
feature_importance = final_model.get_feature_importance()
feature_names = final_available_features

# Create DataFrame and sort by importance
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False).head(30)

# Assign colors based on feature type
colors = []
for feat in importance_df['feature']:
    if feat.startswith('LOT_ID') or '__EQUIP' in feat or '__POSITION' in feat:
        colors.append('red')  # Lot/categorical features
    elif '__SPC_' in feat:
        colors.append('green')  # SPC features
    else:
        colors.append('blue')  # Sensor features

# Plot
plt.figure(figsize=(12, 10))
plt.barh(range(len(importance_df)), importance_df['importance'], color=colors)
plt.yticks(range(len(importance_df)), importance_df['feature'])
plt.xlabel('Feature Importance', fontsize=12)
plt.title(f'Top 30 Features - Lot-Normalized Regression ({vt_type}) - Horizon {final_horizon}', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
importance_file = plots_dir / "regression_lot_normalized_feature_importance.png"
plt.savefig(importance_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"[OK] Saved to {importance_file}")

# Print top 20 features
print("\nTop 20 Most Important Features:")
print("-" * 60)
for i, row in importance_df.head(20).iterrows():
    print(f"{row['feature']:<50} {row['importance']:>8.2f}")
print("-" * 60)

In [ ]:
# Save evaluation summary
print("\n[INFO] Saving evaluation summary...")
summary_file = output_dir / "regression_lot_normalized_evaluation_summary.txt"
with open(summary_file, 'w') as f:
    f.write("=" * 60 + "\n")
    f.write("Lot-Normalized Regression Multi-Horizon Evaluation Summary\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Vt Type: {vt_type}\n")
    f.write(f"Target: Lot-normalized deviation (z-score)\n")
    f.write(f"Final Horizon: {final_horizon}\n")
    f.write(f"Number of Features: {len(final_available_features)}\n")
    f.write(f"Validation Set Size: {len(y_val)}\n\n")
    
    f.write("Performance Metrics (Final Horizon):\n")
    f.write("-" * 40 + "\n")
    f.write(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_pred_final)):.6f}\n")
    f.write(f"MAE:  {mean_absolute_error(y_val, y_pred_final):.6f}\n")
    f.write(f"R²:   {r2_score(y_val, y_pred_final):.4f}\n\n")
    
    f.write("Target Statistics (z-scores):\n")
    f.write("-" * 40 + "\n")
    f.write(f"Mean (actual): {y_val.mean():.6f} (should be ~0)\n")
    f.write(f"Std (actual):  {y_val.std():.6f} (should be ~1)\n")
    f.write(f"Mean (pred):   {y_pred_final.mean():.6f}\n")
    f.write(f"Std (pred):    {y_pred_final.std():.6f}\n\n")
    
    f.write("Horizon Performance Summary:\n")
    f.write("-" * 60 + "\n")
    f.write(f"{'Horizon':<10} {'Features':<10} {'RMSE':<12} {'MAE':<12} {'R²':<10}\n")
    f.write("-" * 60 + "\n")
    for r in horizon_results:
        f.write(f"{r['horizon']:<10} {r['n_features']:<10} {r['rmse']:<12.6f} {r['mae']:<12.6f} {r['r2']:<10.4f}\n")
    f.write("-" * 60 + "\n")

print(f"[OK] Saved to {summary_file}")

In [ ]:
# Mark complete
sentinel.touch()

elapsed = time.time() - start_time
print(f"\n[OK] Lot-normalized regression horizon model training complete in {elapsed:.2f}s")
print("=" * 60)